# ДЗ №3. Задание 3
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Собственные имена по каналам.


## Подготовка

В исходном условии задания 3 оставлен TODO. Здесь используем согласованный вариант: накопление 30 минут, сохранение каждую минуту и итоговый топ-10 отдельно для каждого канала.

Kafka, топик и сборщик те же, что в заданиях 1 и 2. Выбираем ядро ДЗ 3. Если сборщик Telegram уже работает, оставляем его запущенным. Иначе запускаем после последней ячейки. Результаты и checkpoint хранятся в `output`, отдельно для каждого запуска.

In [1]:
import json
import re
import socket
import sys
from datetime import datetime
from operator import add, itemgetter
from pathlib import Path
from threading import Event, Thread

from kafka import KafkaConsumer
from pyspark import SparkConf, SparkContext
from pyspark.streaming import StreamingContext

conf = SparkConf().setMaster("local[2]").setAppName("HW3-Channels")
sc = SparkContext.getOrCreate(conf=conf)
sc.pythonExec = sys.executable
ssc = StreamingContext(sc, 60)

output = (Path("output") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")).resolve()
ssc.checkpoint(str(output / "checkpoint"))

## 1. Поток из Kafka

Читаем новые сообщения из `telegram_messages` через тот же локальный TCP-поток. В JSON уже есть `text` и `channel_id`.

In [2]:
consumer = KafkaConsumer(
    "telegram_messages", bootstrap_servers="localhost:9092",
    auto_offset_reset="latest", enable_auto_commit=False,
)
server = socket.socket()
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("127.0.0.1", 9997))
server.listen(1)
stopped = Event()


def forward_kafka():
    try:
        connection, _ = server.accept()
        with connection:
            while not stopped.is_set():
                records = consumer.poll(timeout_ms=1000)
                for batch in records.values():
                    for record in batch:
                        connection.sendall(record.value + b"\n")
    except (BrokenPipeError, ConnectionResetError):
        pass
    finally:
        consumer.close()
        server.close()


worker = Thread(target=forward_kafka, daemon=True)
lines = ssc.socketTextStream("127.0.0.1", 9997)

## 2. Отбор имён с указанием канала

Как в задании 2, берём русские слова с заглавной буквы, приводим к нижнему регистру и удаляем гласные и «й» с конца. Первое слово предложения учитываем.

Ключ теперь состоит из ID канала и слова. Поэтому одинаковое слово в разных каналах считается отдельно. ID берём из поля `channel_id` сообщения Telegram.

In [3]:
def extract_names(line):
    message = json.loads(line)
    for word in re.findall(r"\b[А-ЯЁ][А-ЯЁа-яё]*\b", message["text"]):
        name = word.lower().rstrip("аеёиоуыэюяй")
        if name:
            yield (message["channel_id"], name), 1

## 3. Накопление и сохранение по каналам

Прибавляем новые вхождения к предыдущим значениям. Каждую минуту сохраняем весь накопленный результат: `ID канала, слово, количество`. В файле строки сгруппированы по каналу, внутри канала — по убыванию частоты.

Данные каждой минуты лежат в `output/<запуск>/<минута>/part-00000`. Для итогового вывода оставляем по 10 самых частых слов каждого канала. При объединении частичных результатов тоже храним только 10 слов.

In [4]:
def update_count(values, previous):
    return (previous or 0) + sum(values)


def sort_key(item):
    (channel_id, word), count = item
    return channel_id, -count


def format_count(item):
    (channel_id, word), count = item
    return f"{channel_id}, {word}, {count}"


def channel_words(item):
    (channel_id, word), count = item
    return channel_id, [(word, count)]


def merge_top(first, second):
    return sorted(first + second, key=itemgetter(1), reverse=True)[:10]


top_by_channel = []


def save_counts(time, rdd):
    global top_by_channel
    ordered = rdd.sortBy(sort_key, numPartitions=1).cache()
    ordered.map(format_count).saveAsTextFile(str(output / time.strftime("%Y%m%d_%H%M%S")))
    top_by_channel = ordered.map(channel_words).reduceByKey(merge_top).collect()
    ordered.unpersist()


minute_counts = lines.flatMap(extract_names).reduceByKey(add)
total_counts = minute_counts.updateStateByKey(update_count)
total_counts.foreachRDD(save_counts)

## 4. Запуск на 30 минут и топ-10 каждого канала

Если сборщик ещё не работает, после запуска этой ячейки запускаем `telegram_producer.py` в отдельном терминале. Через 30 минут поток завершает обработку принятых сообщений и выводит топ-10 каждого канала. Если слов меньше десяти, выводятся все. Каналы без подходящих слов не выводятся.

Сборщик останавливаем через Ctrl+C. Interrupt завершит поток раньше срока. Перед повторным запуском всех ячеек перезапускаем ядро.

In [5]:
worker.start()
ssc.start()
try:
    finish_at = datetime.now().timestamp() + 30 * 60
    while datetime.now().timestamp() < finish_at:
        if ssc.awaitTerminationOrTimeout(1):
            break
except KeyboardInterrupt:
    pass
finally:
    stopped.set()
    ssc.stop(stopSparkContext=False, stopGraceFully=True)

for channel_id, words in sorted(top_by_channel):
    print(f"\nКанал {channel_id}")
    print("Слово, количество")
    for word, count in words:
        print(f"{word}, {count}")


Канал -1001149896996
Слово, количество
госдум, 2
председатель, 1
ирин, 1
белых, 1
ег, 1
олимпиад, 1
согласн, 1

Канал -1001054549314
Слово, количество
в, 1
запрет, 1
хакас, 1
подписатьс, 1
нтв, 1
макс, 1

Канал -1001050820672
Слово, количество
тасс, 5
макс, 3
в, 2
мвд, 2
ирин, 2
подпишись, 2
волк, 2
нат, 1
сф, 1
такж, 1

Канал -1001036362176
Слово, количество
подписатьс, 3
рассылк, 3
сайт, 3
он, 2
мужчин, 1
ижевск, 1
вид, 1
европ, 1
ран, 1
франц, 1

Канал -1001001872252
Слово, количество
владимир, 3
кмс, 3
михаил, 2
соловьёв, 2
чтоб, 2
чибис, 2
дегтярев, 2
ис, 2
вест, 2
сайт, 2
